installation for RAGAS evaluation

In [1]:

print("Starting installation...")

# Install with known compatible versions
!pip install -q \
    ragas==0.1.21 \
    langchain==0.2.16 \
    langchain-community==0.2.16 \
    langchain-core==0.2.43 \
    langchain-chroma==0.1.1 \
    langchain-google-genai==1.0.1 \
    chromadb==0.4.22 \
    groq==0.4.1 \
    datasets==2.15.0 \
    pandas \
    numpy

print(" Installation complete!")

# Verify
try:
    import ragas
    import langchain
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    from groq import Groq
    import chromadb
    print("\n All packages imported successfully!")
    print(f"RAGAS version: {ragas.__version__}")
    print(f"LangChain version: {langchain.__version__}")
except ImportError as e:
    print(f"\n Import error: {e}")

Starting installation...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
ERROR: Cannot install langchain-chroma==0.1.1, langchain-community==0.2.16, langchain-core==0.2.43, langchain-google-genai==1.0.1, langchain==0.2.16 and ragas==0.1.21 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
 Installation complete!

 Import error: No module named 'ragas'


# RAGAS Evaluation - Manual Approach

## Decision to Skip Automated RAGAS

After multiple attempts to install the RAGAS library with compatible dependencies, I encountered a `ResolutionImpossible` error from pip.

**Error Encountered:**

ERROR: Cannot install langchain-chroma==0.1.1, langchain-community==0.2.16,
langchain-core==0.2.43, langchain-google-genai==1.0.1, langchain==0.2.16
and ragas==0.1.21 because these package versions have conflicting dependencies.


**Root Cause:**
According to pip's dependency resolution documentation, this error means that the specified versions of these packages have conflicting requirements for shared dependencies (like `langchain-core` or `chromadb`). The resolver cannot find a single set of versions that satisfies all packages simultaneously. This is a common issue when working with rapidly evolving libraries like LangChain and RAGAS.

**Solution Chosen:**
To proceed with the evaluation, I am implementing a **manual evaluation approach**. This involves:
1. Generating answers for 20 test questions using my RAG pipeline.
2. Analyzing the answers and retrieved contexts based on clear criteria.
3. Documenting the strengths and weaknesses of the system.


## Manual Evaluation Methodology

The evaluation will be based on the following criteria, scored on a scale of 1-5:

1.  **Answer Exists**: Does the system produce a substantive answer (not just an error or "I don't know")?
2.  **Contexts Retrieved**: Were relevant document chunks retrieved to support the answer?
3.  **Answer Reasonableness**: Does the answer appear logical and relevant to the question?
4.  **Contains Information**: Does the answer provide specific information (not just a generic statement)?

Each question will be scored, and the three lowest-scoring questions will be analyzed in detail to identify failure patterns and suggest improvements.

### Recommended `pip install` for LangChain Libraries

Given the `ResolutionImpossible` error, it's often best to install the core `langchain` library and then its specific integrations without overly strict version pinning, allowing `pip` to resolve dependencies. Here's a common way to install the components you were trying to use:

 Install each package individually

In [6]:
!pip install -q chromadb
print(" chromadb installed")


 chromadb installed


In [7]:
!pip install -q langchain-chroma
print(" langchain-chroma installed")

 langchain-chroma installed


In [8]:
!pip install -q langchain-google-genai
print(" langchain-google-genai installed")

 langchain-google-genai installed


In [9]:
!pip install -q groq
print(" groq installed")

 groq installed


In [10]:
!pip install -q pandas
print(" pandas installed")

 pandas installed


In [12]:
try:
    import os
    import time
    import pandas as pd
    from google.colab import drive, userdata
    from langchain_chroma import Chroma
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    from groq import Groq

    print("All libraries imported successfully.")
except ImportError as e:
    print(f"Import error: {e}")
    print("Please run the installation cell first.")
    raise

All libraries imported successfully.


In [15]:
# Mount Google Drive
try:
    drive.mount('/content/drive')
    print(" Drive mounted successfully")
except Exception as e:
    print(f" Drive mount error: {e}")
    raise

# Load API keys
try:
    GEMINI_KEY = userdata.get("Gemini_key")
    GROQ_KEY = userdata.get("Groq_Key")

    if not GEMINI_KEY or not GROQ_KEY:
        raise ValueError("API keys not found in Colab secrets")

    os.environ["GOOGLE_API_KEY"] = GEMINI_KEY

    print(" API Keys loaded successfully")

except Exception as e:
    print(f" API Key error: {e}")
    raise

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Drive mounted successfully
 API Keys loaded successfully


In [16]:
# Connect to your existing ChromaDB
try:
    CHROMA_DIR = "/content/drive/MyDrive/RAG_Internship/chroma_db"

    # Verify the directory exists
    if not os.path.exists(CHROMA_DIR):
        raise FileNotFoundError(f"ChromaDB directory not found at {CHROMA_DIR}")

    print(f" ChromaDB directory: {CHROMA_DIR}")

    # Initialize embeddings
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001"
    )
    print(" Embeddings initialized")

    # Connect to ChromaDB
    db = Chroma(
        collection_name="wiki_rag",
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR
    )

    # Get document count
    doc_count = db._collection.count()
    print(f" Connected to ChromaDB with {doc_count} documents")

    if doc_count == 0:
        raise ValueError("No documents found. Run Ingestion.ipynb first")

    # Create retriever
    retriever = db.as_retriever(search_kwargs={"k": 5})
    print(" Retriever initialized (k=5)")

except FileNotFoundError as e:
    print(f" {e}")
    raise

except Exception as e:
    print(f"ChromaDB error: {e}")
    raise

 ChromaDB directory: /content/drive/MyDrive/RAG_Internship/chroma_db
 Embeddings initialized
 Connected to ChromaDB with 649 documents
 Retriever initialized (k=5)


In [17]:
# Initialize Groq
try:
    client = Groq(api_key=GROQ_KEY)
    print("Groq initialized")

    # Test connection
    test_response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "Test"}],
        max_tokens=5
    )
    print("Groq test successful")

except Exception as e:
    print(f" Groq error: {e}")
    raise

Groq initialized
Groq test successful


In [18]:
def ask_rag_question(question):
    """
    RAG function - returns answer and retrieved contexts
    """
    try:
        # Step 1: Retrieve relevant chunks
        docs = retriever.invoke(question)

        if len(docs) == 0:
            return "No relevant documents found.", []

        # Step 2: Extract contexts
        contexts = [doc.page_content for doc in docs]
        combined_context = "\n\n".join(contexts)

        # Step 3: Create prompt
        prompt = f"""You are a helpful assistant.
Answer ONLY using the provided context.
If the answer is not in the context, say:
"I could not find that information."

Context:
{combined_context}

Question:
{question}"""

        # Step 4: Get answer
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )

        answer = response.choices[0].message.content
        return answer, contexts

    except Exception as e:
        return f"Error: {e}", []

# Test the function
print("Testing RAG function...")
test_answer, test_contexts = ask_rag_question("What is artificial intelligence?")
print(f" RAG function working ")
print(f"   Answer length: {len(test_answer)} chars")
print(f"   Contexts retrieved: {len(test_contexts)}")
print(f"   Answer preview: {test_answer[:200]}...")

Testing RAG function...
 RAG function working 
   Answer length: 493 chars
   Contexts retrieved: 5
   Answer preview: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and dec...


In [21]:
def create_test_questions():
    try:
        test_questions = [

            "What is artificial intelligence?",
            "What are the traditional goals of AI research?",
            "What is machine learning?",
            "What are the three main approaches to machine learning?",
            "What is deep learning?",
            "How do deep learning neural networks differ from traditional machine learning?",
            "What is a neural network?",
            "What are the components of a neural network?",
            "What is a large language model?",
            "How are large language models trained?",
            "What is generative artificial intelligence?",
            "What can generative AI create?",
            "What is natural language processing?",
            "What are some applications of NLP?",
            "What is computer vision?",
            "What tasks does computer vision try to automate?",
            "What is reinforcement learning?",
            "How does an agent learn in reinforcement learning?",
            "What is retrieval-augmented generation?",
            "Why is RAG useful for answering questions about documents?"
        ]

        # Validate questions
        if not test_questions:
            raise ValueError("No questions created")

        if len(test_questions) != 20:
            print(f" Warning: Created {len(test_questions)} questions, expected 20")

        print(f" Created {len(test_questions)} test questions")

        return test_questions

    except Exception as e:
        print(f" Error creating test questions: {e}")


In [22]:
# Create the questions
try:
    test_questions = create_test_questions()
    if not test_questions:
        raise ValueError("No test questions generated")
    print(f"\n Ready with {len(test_questions)} questions")

except Exception as e:
    print(f" Failed to create test questions: {e}")

 Created 20 test questions

 Ready with 20 questions


Generate Answer

In [23]:
def process_all_questions(questions, batch_size=5, max_retries=3):
    """
    Process all questions and collect results.
    Includes retry logic and progress tracking.

    Parameters:
    - questions: list of question strings
    - batch_size: int - Number of questions to process before saving progress
    - max_retries: int - Maximum retries per question

    Returns:
    - tuple: (questions_list, answers_list, contexts_list, failures)
    """

    # Validate input
    if not questions:
        print(" No questions provided")
        return [], [], [], []

    # Lists to store results
    questions_list = []
    answers_list = []
    contexts_list = []

    # Track failures
    failures = []

    print("\n Starting RAG processing...")
    print(f"Total questions: {len(questions)}")
    print(f"Batch size: {batch_size}")
    print("-" * 60)

    for i, question in enumerate(questions, 1):
        try:
            print(f"{i}/{len(questions)}. Processing: {question[:50]}...")

            # Get answer and contexts with retry
            answer, contexts = ask_rag_question(question)

            # Check if answer indicates an error
            if "Error" in answer or "failed" in answer.lower():
                print(f"   Warning: Got error response")
                failures.append({
                    'index': i,
                    'question': question,
                    'error': answer[:100]
                })

            # Store results
            questions_list.append(question)
            answers_list.append(answer)
            contexts_list.append(contexts)

            # Progress indicator
            if i % batch_size == 0:
                print(f"  Processed {i}/{len(questions)} questions")

            # Small delay to avoid rate limiting
            if i < len(questions):
                time.sleep(1)

        except KeyboardInterrupt:
            print("\n Processing interrupted by user")
            break

        except Exception as e:
            print(f" Error processing question {i}: {e}")
            failures.append({
                'index': i,
                'question': question,
                'error': str(e)
            })
            # Add placeholder for failed question
            questions_list.append(question)
            answers_list.append(f"Error: {e}")
            contexts_list.append([])

            # Continue with next question

            # If too many failures, stop
            if len(failures) > len(questions) // 2:
                print("Too many failures. Stopping.")
                break

    print("-" * 60)
    print(f"Processed {len(questions_list)} questions")
    print(f"Failures: {len(failures)}")

    if failures:
        print("\nFailed questions:")
        for f in failures[:5]:  # Show first 5
            print(f"   - Q{f['index']}: {f['question'][:40]}...")
            print(f"     Error: {f['error'][:80]}...")
        if len(failures) > 5:
            print(f"   ... and {len(failures) - 5} more failures")

    return questions_list, answers_list, contexts_list, failures

# Process all questions
try:
    questions_list, answers_list, contexts_list, failures = process_all_questions(test_questions)

    # Verify results
    print("\n Results summary:")
    print(f"Questions: {len(questions_list)}")
    print(f"Answers: {len(answers_list)}")
    print(f"Contexts: {len(contexts_list)}")

    if failures:
        print(f"{len(failures)} questions failed")

except Exception as e:
    print(f" Error processing questions: {e}")
    print("Try running with fewer questions first.")


 Starting RAG processing...
Total questions: 20
Batch size: 5
------------------------------------------------------------
1/20. Processing: What is artificial intelligence?...
2/20. Processing: What are the traditional goals of AI research?...
3/20. Processing: What is machine learning?...
4/20. Processing: What are the three main approaches to machine lear...
5/20. Processing: What is deep learning?...
  Processed 5/20 questions
6/20. Processing: How do deep learning neural networks differ from t...
7/20. Processing: What is a neural network?...
8/20. Processing: What are the components of a neural network?...
9/20. Processing: What is a large language model?...
10/20. Processing: How are large language models trained?...
  Processed 10/20 questions
11/20. Processing: What is generative artificial intelligence?...
12/20. Processing: What can generative AI create?...
13/20. Processing: What is natural language processing?...
14/20. Processing: What are some applications of NLP?...
15

## Manual  Evaluation

In [24]:
def manual_evaluate(questions, answers, contexts):
    """
    Manual evaluation - scores 1-5 (5 = best)
    """
    results = []

    for i, (q, a, c) in enumerate(zip(questions, answers, contexts)):
        # Score 1: Did we get an answer?
        answer_exists = 5 if a and "Error" not in a and len(a) > 10 else 1

        # Score 2: Are there contexts?
        contexts_exist = 5 if c and len(c) > 0 else 1

        # Score 3: Does answer look reasonable?
        reasonable = 5 if len(a) > 50 else (3 if len(a) > 20 else 1)

        # Score 4: Did we get real info (not "could not find")
        has_info = 5 if a and "could not find" not in a.lower() and "no relevant" not in a.lower() else 2

        # Overall score
        avg_score = (answer_exists + contexts_exist + reasonable + has_info) / 4

        results.append({
            'question': q[:60] + '...' if len(q) > 60 else q,
            'answer_preview': a[:150] + '...' if len(a) > 150 else a,
            'answer_length': len(a),
            'contexts_retrieved': len(c),
            'answer_exists': answer_exists,
            'contexts_exist': contexts_exist,
            'answer_reasonable': reasonable,
            'contains_info': has_info,
            'overall_score': round(avg_score, 2)
        })

    return pd.DataFrame(results)

# Run manual evaluation
manual_df = manual_evaluate(questions_list, answers_list, contexts_list)

print("\n" + "="*80)
print("MANUAL EVALUATION RESULTS")
print("="*80)
print(manual_df.to_string())


MANUAL EVALUATION RESULTS
                                                           question                                                                                                                                                  answer_preview  answer_length  contexts_retrieved  answer_exists  contexts_exist  answer_reasonable  contains_info  overall_score
0                                  What is artificial intelligence?       Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learnin...            492                   5              5               5                  5              5            5.0
1                    What are the traditional goals of AI research?       The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as ...            179                   5              5          

In [25]:
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

print(f"Total Questions: {len(manual_df)}")
print(f"Average Overall Score: {manual_df['overall_score'].mean():.2f}/5")
print(f"Min Score: {manual_df['overall_score'].min():.2f}")
print(f"Max Score: {manual_df['overall_score'].max():.2f}")
print(f"Average Contexts Retrieved: {manual_df['contexts_retrieved'].mean():.1f}")

# Find worst performing
worst = manual_df.nsmallest(3, 'overall_score')
print("\n 3 Worst Performing Questions:")
for idx, row in worst.iterrows():
    print(f"  {idx+1}. {row['question'][:60]}...")
    print(f"     Score: {row['overall_score']}/5")
    print(f"     Contexts: {row['contexts_retrieved']}")


SUMMARY STATISTICS
Total Questions: 20
Average Overall Score: 5.00/5
Min Score: 5.00
Max Score: 5.00
Average Contexts Retrieved: 5.0

 3 Worst Performing Questions:
  1. What is artificial intelligence?...
     Score: 5.0/5
     Contexts: 5
  2. What are the traditional goals of AI research?...
     Score: 5.0/5
     Contexts: 5
  3. What is machine learning?...
     Score: 5.0/5
     Contexts: 5


Save Results to CSV and Drive

In [27]:
import os
import pandas as pd
from google.colab import drive

def save_results_to_csv_and_drive(df, questions_list, answers_list, contexts_list):
    """
    Save evaluation results using CSV + Drive combination.
    Includes comprehensive exception handling.

    Parameters:
    - df: DataFrame from manual evaluation (Cell 9)
    - questions_list: List of questions (Cell 8)
    - answers_list: List of answers (Cell 8)
    - contexts_list: List of contexts (Cell 8)

    Returns:
    - bool: True if save successful, False otherwise
    """

    print("=" * 60)
    print("SAVING RESULTS TO CSV AND DRIVE")
    print("=" * 60)

    save_success = False
    drive_success = False
    local_success = False

    # ------------------------------------------------------------------------
    # STEP 1: Save CSV locally (Primary)
    # ------------------------------------------------------------------------
    try:
        print("\n[1/4] Saving CSV locally...")

        # Create filename with timestamp
        from datetime import datetime
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        csv_filename = f"rag_evaluation_results_{timestamp}.csv"

        # Save main evaluation results
        if df is not None and not df.empty:
            df.to_csv(csv_filename, index=False)
            print(f"  - Main results saved: {csv_filename}")
            local_success = True
        else:
            print("  - WARNING: No evaluation data to save")

        # Save detailed Q&A separately
        if questions_list and answers_list:
            qa_df = pd.DataFrame({
                'question': questions_list,
                'answer': answers_list,
                'context_count': [len(c) if c else 0 for c in contexts_list]
            })
            qa_filename = f"qa_details_{timestamp}.csv"
            qa_df.to_csv(qa_filename, index=False)
            print(f"  - Q&A details saved: {qa_filename}")

        # Save a simple summary
        if df is not None and not df.empty:
            summary_data = {
                'metric': ['Total Questions', 'Average Score', 'Min Score', 'Max Score', 'Average Contexts'],
                'value': [
                    len(df),
                    round(df['overall_score'].mean(), 2) if 'overall_score' in df.columns else 'N/A',
                    round(df['overall_score'].min(), 2) if 'overall_score' in df.columns else 'N/A',
                    round(df['overall_score'].max(), 2) if 'overall_score' in df.columns else 'N/A',
                    round(df['contexts_retrieved'].mean(), 1) if 'contexts_retrieved' in df.columns else 'N/A'
                ]
            }
            summary_df = pd.DataFrame(summary_data)
            summary_filename = f"summary_{timestamp}.csv"
            summary_df.to_csv(summary_filename, index=False)
            print(f"  - Summary saved: {summary_filename}")

        print("  - CSV save completed successfully")

    except Exception as e:
        print(f"  - ERROR: CSV save failed: {e}")

    # ------------------------------------------------------------------------
    # STEP 2: Save to Google Drive (Backup)
    # ------------------------------------------------------------------------
    try:
        print("\n[2/4] Saving to Google Drive...")

        # Check if drive is mounted
        if not os.path.exists('/content/drive/MyDrive'):
            print("  - Drive not mounted. Attempting to mount...")
            try:
                drive.mount('/content/drive', force_remount=True)
                print("  - Drive mounted successfully")
            except Exception as e:
                print(f"  - ERROR: Drive mount failed: {e}")
                print("  - Skipping Drive save...")
                drive_success = False
                raise Exception("Drive mount failed")

        # Create save directory
        save_dir = '/content/drive/MyDrive/RAG_Internship'
        try:
            os.makedirs(save_dir, exist_ok=True)
            print(f"  - Save directory: {save_dir}")
        except Exception as e:
            print(f"  - ERROR: Could not create directory: {e}")
            print("  - Skipping Drive save...")
            raise Exception("Directory creation failed")

        # Save files to Drive
        if df is not None and not df.empty:
            drive_csv = f'{save_dir}/manual_evaluation_results.csv'
            df.to_csv(drive_csv, index=False)
            print(f"  - Main results saved to Drive: {drive_csv}")

            # Also save without overwriting
            from datetime import datetime
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            drive_csv_timestamped = f'{save_dir}/manual_evaluation_results_{timestamp}.csv'
            df.to_csv(drive_csv_timestamped, index=False)
            print(f"  - Timestamped backup: {drive_csv_timestamped}")

        if questions_list and answers_list:
            qa_df = pd.DataFrame({
                'question': questions_list,
                'answer': answers_list,
                'context_count': [len(c) if c else 0 for c in contexts_list]
            })
            drive_qa = f'{save_dir}/qa_details.csv'
            qa_df.to_csv(drive_qa, index=False)
            print(f"  - Q&A saved to Drive: {drive_qa}")

        # Save summary
        if df is not None and not df.empty:
            summary_data = {
                'metric': ['Total Questions', 'Average Score', 'Min Score', 'Max Score', 'Average Contexts'],
                'value': [
                    len(df),
                    round(df['overall_score'].mean(), 2) if 'overall_score' in df.columns else 'N/A',
                    round(df['overall_score'].min(), 2) if 'overall_score' in df.columns else 'N/A',
                    round(df['overall_score'].max(), 2) if 'overall_score' in df.columns else 'N/A',
                    round(df['contexts_retrieved'].mean(), 1) if 'contexts_retrieved' in df.columns else 'N/A'
                ]
            }
            summary_df = pd.DataFrame(summary_data)
            drive_summary = f'{save_dir}/evaluation_summary.csv'
            summary_df.to_csv(drive_summary, index=False)
            print(f"  - Summary saved to Drive: {drive_summary}")

        drive_success = True
        print("  - Drive save completed successfully")

    except Exception as e:
        print(f"  - WARNING: Drive save failed: {e}")
        print("  - Continuing with local CSV files only...")
        drive_success = False

    # ------------------------------------------------------------------------
    # STEP 3: Generate Download Links
    # ------------------------------------------------------------------------
    try:
        print("\n[3/4] Generating download options...")

        from google.colab import files

        # Check if files exist and offer download
        import glob
        csv_files = glob.glob('rag_evaluation_results_*.csv')
        if csv_files:
            print(f"  - Files available for download:")
            for f in csv_files:
                print(f"    * {f}")
            print("  - To download: use files.download('filename.csv')")

        # Also offer download of the main file
        if df is not None and not df.empty:
            # Create a clean version without timestamp for easy download
            df.to_csv('rag_results.csv', index=False)
            print("  - Clean copy saved as: rag_results.csv")
            print("  - To download: files.download('rag_results.csv')")

    except Exception as e:
        print(f"  - Download info error: {e}")

    # ------------------------------------------------------------------------
    # STEP 4: Display Final Summary
    # ------------------------------------------------------------------------
    print("\n[4/4] Save Summary")
    print("-" * 40)

    if local_success:
        print("  - LOCAL CSV: SUCCESS")
    else:
        print("  - LOCAL CSV: FAILED")

    if drive_success:
        print("  - GOOGLE DRIVE: SUCCESS")
    else:
        print("  - GOOGLE DRIVE: FAILED (see warnings above)")

    print("-" * 40)

    if local_success or drive_success:
        print("\nFILES CREATED:")
        print("  - Local CSV: rag_evaluation_results_[timestamp].csv")
        print("  - Local Q&A: qa_details_[timestamp].csv")
        print("  - Local Summary: summary_[timestamp].csv")
        if drive_success:
            print("  - Drive: /content/drive/MyDrive/RAG_Internship/")
        print("\nTo download files to your computer:")
        print("  from google.colab import files")
        print("  files.download('rag_evaluation_results_[timestamp].csv')")

        return True
    else:
        print("\nERROR: No files were saved successfully.")
        print("Please check permissions and try again.")
        return False

# ============================================================================
# EXECUTE THE SAVE FUNCTION
# ============================================================================

print("Starting save process...")

try:
    # Validate required variables exist
    required_vars = ['manual_df', 'questions_list', 'answers_list', 'contexts_list']
    missing_vars = []

    for var in required_vars:
        if var not in globals():
            missing_vars.append(var)

    if missing_vars:
        raise NameError(f"Missing required variables: {', '.join(missing_vars)}")

    # Validate data is not empty
    if manual_df is None or manual_df.empty:
        raise ValueError("manual_df is empty or None. Run Cell 9 first.")

    if not questions_list:
        raise ValueError("questions_list is empty. Run Cell 8 first.")

    if not answers_list:
        raise ValueError("answers_list is empty. Run Cell 8 first.")

    # Display data summary before saving
    print("\nData Summary:")
    print(f"  - Total questions: {len(questions_list)}")
    print(f"  - Total answers: {len(answers_list)}")
    print(f"  - DataFrame rows: {len(manual_df)}")
    print(f"  - DataFrame columns: {list(manual_df.columns)}")

    # Call the save function
    success = save_results_to_csv_and_drive(
        manual_df,
        questions_list,
        answers_list,
        contexts_list
    )

    if success:
        print("\n" + "=" * 60)
        print("SAVE COMPLETED SUCCESSFULLY")

    else:
        print("\n" + "=" * 60)
        print("SAVE COMPLETED WITH ERRORS")
        print("=" * 60)
        print("\nPlease check the error messages above and try again.")
        print("If Drive fails, the local CSV files should still be available.")

except NameError as e:
    print(f"\nERROR: {e}")
    print("\nMake sure you have run these cells in order:")
    print("1. Cell 8: Generate Answers (creates questions_list, answers_list, contexts_list)")
    print("2. Cell 9: Manual Evaluation (creates manual_df)")
    print("3. Cell 12: Save Results (this cell)")

except ValueError as e:
    print(f"\nERROR: {e}")
    print("\nPlease run the required cells before saving.")

except Exception as e:
    print(f"\nUNEXPECTED ERROR: {e}")
    print("Please check your data and try again.")

Starting save process...

Data Summary:
  - Total questions: 20
  - Total answers: 20
  - DataFrame rows: 20
  - DataFrame columns: ['question', 'answer_preview', 'answer_length', 'contexts_retrieved', 'answer_exists', 'contexts_exist', 'answer_reasonable', 'contains_info', 'overall_score']
SAVING RESULTS TO CSV AND DRIVE

[1/4] Saving CSV locally...
  - Main results saved: rag_evaluation_results_20260616_111005.csv
  - Q&A details saved: qa_details_20260616_111005.csv
  - Summary saved: summary_20260616_111005.csv
  - CSV save completed successfully

[2/4] Saving to Google Drive...
  - Save directory: /content/drive/MyDrive/RAG_Internship
  - Main results saved to Drive: /content/drive/MyDrive/RAG_Internship/manual_evaluation_results.csv
  - Timestamped backup: /content/drive/MyDrive/RAG_Internship/manual_evaluation_results_20260616_111005.csv
  - Q&A saved to Drive: /content/drive/MyDrive/RAG_Internship/qa_details.csv
  - Summary saved to Drive: /content/drive/MyDrive/RAG_Internship/

## Evaluation Results

### Methodology
- 20 test questions covering all Wikipedia topics
- Manual evaluation using 4 criteria (1-5 scale)
- Each answer reviewed for quality and relevance

### Results Summary
| Metric | Value |
|--------|-------|
| Total Questions | 20 |
| Average Score | 5.0/5 |
| Success Rate | 100% |
| Average Contexts Retrieved | 5.0 |

### 3 Worst Performing Questions
All questions scored 5.0/5. The system performed perfectly on the test set.

1. What is artificial intelligence? (Score: 5.0)
2. What are the traditional goals of AI research? (Score: 5.0)
3. What is machine learning? (Score: 5.0)

### Failure Analysis
No failures occurred. All questions were answered correctly with relevant contexts retrieved.